a)	What are vanishing and exploding gradients? How do they affect neural networks? Explain in detail.

During the training of deep neural networks using **backpropagation**, gradients are calculated by the chain rule, moving from the output layer back toward the input layer. These gradients are used to update the weights of the network.

#### 1. Vanishing Gradients
*   **What it is**: This occurs when the gradients of the loss function with respect to the weights become extremely small (close to zero) as they are propagated backward through many layers.
*   **Why it happens**: It is common when using activation functions like **Sigmoid** or **Tanh**, which have derivatives between 0 and 1. Multiplying many such small numbers across layers causes the gradient to decrease exponentially.
*   **Effect**: The weights in the earlier layers (near the input) change very little or not at all. Consequently, the network stops learning, and training stalls before the model converges to an optimal solution.

#### 2. Exploding Gradients
*   **What it is**: The opposite of vanishing gradients; here, the gradients grow exponentially large as they propagate backward through the network.
*   **Why it happens**: This typically occurs due to large initial weights or certain network architectures where the product of the gradients and weights exceeds 1 at each layer.
*   **Effect**: The weight updates become massive, causing the model's weights to oscillate wildly or reach 'NaN' (Not a Number) values. This leads to an unstable network that fails to converge.

#### Summary of Impacts:
*   **Training Time**: Both issues significantly increase training time or make it impossible for the model to learn complex patterns.
*   **Architecture Limits**: These problems originally limited how deep a neural network could be until solutions like **ReLU** activation, **Batch Normalization**, and **Residual Connections** (ResNets) were introduced.

b)	Use the Bank customer churn dataset from below Kaggle link and create an end-to-end project on Jupyter/Colab to predict the Churn.

https://www.kaggle.com/datasets/santoshd3/bank-customers/data

i.	Download the dataset from above link and load it into your Python environment.

ii.	Perform the EDA and do the visualizations.

iii.	Check the distributions/skewness in the variables and do the transformations if required.

iv.	Check/Treat the outliers and do the feature scaling if required.

v.	Build Deep Learning model using ANN with multiple hidden layers.

vi.	Apply the dropout regularization and early stopping techniques to improve model performance.

vii.	Use the modelCheckpoint also to store the parameters after each epoch.

viii.	Use the KerasTuner to tune to best parameters (No. of hidden layers, optimizers, loss function, activation functions etc.)

ix.	Compare the accuracies of different models and finalize the best model.


IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# TensorFlow and Keras Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

try:
    import keras_tuner as kt
except ImportError:
    !pip install keras-tuner -q
    import keras_tuner as kt

# Visual Settings
%matplotlib inline
sns.set(style='whitegrid')
warnings.filterwarnings('ignore')
print("Libraries and dependencies imported successfully.")

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

LOAD DATASET

In [ ]:
# Load the dataset
Bank_Cust_Churn_Data = pd.read_csv('Bank_Customer_Churn.csv')

print("--- Dataset Shape ---")
print(Bank_Cust_Churn_Data.shape)

print("\n--- Head ---")
display(Bank_Cust_Churn_Data.head())

print("\n--- Data Info ---")
Bank_Cust_Churn_Data.info()

print("\n--- Missing Values ---")
print(Bank_Cust_Churn_Data.isnull().sum())

print("\n--- Duplicate Rows ---")
print(Bank_Cust_Churn_Data.duplicated().sum())

print("\n--- Statistical Summary ---")
display(Bank_Cust_Churn_Data.describe().T)

EDA :Target Distribution and Feature Analysis

In [ ]:
plt.figure(figsize=(15, 10))

# 1. Target Distribution
plt.subplot(2, 2, 1)
sns.countplot(x='Exited', data=Bank_Cust_Churn_Data, palette='viridis')
plt.title('Churn Distribution (0=Stayed, 1=Exited)')

# 2. Correlation Heatmap
plt.subplot(2, 2, 2)
numeric_df = Bank_Cust_Churn_Data.select_dtypes(include=[np.number]).drop(['RowNumber', 'CustomerId'], axis=1)
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')

# 3. Age Distribution by Churn
plt.subplot(2, 2, 3)
sns.histplot(data=Bank_Cust_Churn_Data, x='Age', hue='Exited', kde=True, element='step')
plt.title('Age Distribution vs Churn')

# 4. Geography vs Churn
plt.subplot(2, 2, 4)
sns.countplot(x='Geography', hue='Exited', data=Bank_Cust_Churn_Data, palette='magma')
plt.title('Geography vs Churn')

plt.tight_layout()
plt.show()

DATA PREPROCESSING : Handling encoding, outliers, scaling, skewness, and splitting.

In [ ]:
# 1. Drop Irrelevant Columns
Bank_Cust_Churn_Data = Bank_Cust_Churn_Data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
print("Irrelevant Columns Dropped.")

# 2. Categorical Encoding
le = LabelEncoder()
Bank_Cust_Churn_Data['Gender'] = le.fit_transform(Bank_Cust_Churn_Data['Gender'])
Bank_Cust_Churn_Data = pd.get_dummies(Bank_Cust_Churn_Data, columns=['Geography'], drop_first=True)
print("Categorical Encodeing Done.")

# 3. Check for Skewness before capping
print("--- Skewness in Numerical Features ---")
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
print(Bank_Cust_Churn_Data[numerical_cols].skew())

# 4. Outlier Treatment (Capping)
for col in ['Age', 'CreditScore']:
    Q1 = Bank_Cust_Churn_Data[col].quantile(0.25)
    Q3 = Bank_Cust_Churn_Data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    Bank_Cust_Churn_Data[col] = np.clip(Bank_Cust_Churn_Data[col], lower, upper)

print("Outlier Treatment Done.")

# 5. Train-Test Split (BEFORE Scaling to prevent leakage)
X = Bank_Cust_Churn_Data.drop('Exited', axis=1)
y = Bank_Cust_Churn_Data['Exited']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train-Test Split Done.")

# 6. Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
print("Feature Scaling Done.")

print("--- Final Training Data Shape ---")
print(f"Final Training Shape: {X_train.shape}")

Evaluating ML models

In [ ]:
# Model Dictionary
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier()
}

def evaluate_models(X_train, X_test, y_train, y_test, models, label):
    perf_list = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred_test = model.predict(X_test)
        y_prob_test = model.predict_proba(X_test)[:, 1]
        y_pred_train = model.predict(X_train)

        perf_list.append({
            "Model": name,
            "Data Type": label,
            "Train Accuracy": accuracy_score(y_train, y_pred_train),
            "Test Accuracy": accuracy_score(y_test, y_pred_test),
            "Precision": precision_score(y_test, y_pred_test),
            "Recall": recall_score(y_test, y_pred_test),
            "F1 Score": f1_score(y_test, y_pred_test),
            "ROC-AUC": roc_auc_score(y_test, y_prob_test)
        })

    return pd.DataFrame(perf_list).sort_values(by='Test Accuracy', ascending=False)

print("Models and evaluation function updated to include training accuracy.")

Evaluating on Imbalanced Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Running evaluation on imbalanced data
results_imbalanced = evaluate_models(X_train, X_test, y_train, y_test, models, "Imbalanced")

print("--- Experiment 1 Results (Imbalanced Data) ---")
display(results_imbalanced)

Handling Data Imbalane

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Note: SMOTE should only be applied to the training data to avoid data leakage
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("--- Class Distribution After SMOTE ---")
print(f"Original training set shape: {Counter(y_train)}")
print(f"Resampled training set shape: {Counter(y_train_res)}")

Evaluating on Balanced Dataset

In [ ]:
# Running evaluation on balanced data
results_balanced = evaluate_models(X_train_res, X_test, y_train_res, y_test, models, "Balanced")

print("--- Experiment 2 Results (Balanced Data) ---")
display(results_balanced)

MODEL SELECTION

Based on the comparison between imbalanced and balanced experiments, we select the top 2 models that provide the best trade-off between ROC-AUC and Recall on the balanced data.

In [ ]:
# 1. Merge the results from both experiments
combined_results = pd.concat([results_imbalanced, results_balanced], axis=0).reset_index(drop=True)

print("--- Combined Model Performance ---")
display(combined_results.sort_values(by='Test Accuracy', ascending=False))

# 2. Dynamically select the top 2 models based on Test Accuracy from the balanced experiment
top_2_models_info = results_balanced.head(2)
top_model_names = top_2_models_info['Model'].tolist()

print("\n--- Top 2 Selected Models (from Balanced Data) ---")
for i, name in enumerate(top_model_names, 1):
    acc = top_2_models_info.iloc[i-1]['Test Accuracy']
    print(f"{i}. {name} (Accuracy: {acc:.4f})")

Hyperparameter Tuning for Top ML Models

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Hyperparameter Tuning for LightGBM
print("Starting Hyperparameter Tuning for LightGBM...")
lgbm_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'num_leaves': [31, 50],
    'boosting_type': ['gbdt']
}

lgbm_grid = GridSearchCV(LGBMClassifier(random_state=42, verbose=-1), lgbm_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
lgbm_grid.fit(X_train_res, y_train_res)

print(f"Best LightGBM Params: {lgbm_grid.best_params_}")
print(f"Best LightGBM CV Accuracy: {lgbm_grid.best_score_:.4f}")

# 2. Hyperparameter Tuning for XGBoost
print("\nStarting Hyperparameter Tuning for XGBoost...")
xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6],
    'eval_metric': ['logloss']
}

xgb_grid = GridSearchCV(XGBClassifier(random_state=42), xgb_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X_train_res, y_train_res)

print(f"Best XGBoost Params: {xgb_grid.best_params_}")
print(f"Best XGBoost CV Accuracy: {xgb_grid.best_score_:.4f}")

# 3. Store tuned results for final comparison
tuned_lgbm_acc = accuracy_score(y_test, lgbm_grid.predict(X_test))
tuned_xgb_acc = accuracy_score(y_test, xgb_grid.predict(X_test))

print(f"\nTuned LightGBM Test Accuracy: {tuned_lgbm_acc:.4f}")
print(f"Tuned XGBoost Test Accuracy: {tuned_xgb_acc:.4f}")

Building & Training the ANN

We will now construct a Sequential ANN model with:
- Two hidden layers (32 and 16 neurons).
- Dropout layers (20%) for regularization.
- Early Stopping and Model Checkpoint callbacks to ensure optimal training.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Define the ANN architecture
model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Set up callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=1000,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy without Keras Tuner: {test_accuracy}")


In [ ]:
y_pred=model.predict(X_test)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    # Tune the number of hidden layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=16, max_value=64, step=16),
            activation=hp.Choice('activation', ['relu', 'tanh', 'sigmoid'])
        ))
        model.add(layers.Dropout(hp.Float('dropout', 0, 0.5, step=0.1)))

    model.add(layers.Dense(1, activation='sigmoid'))

    # Tune the optimizer and loss function
    optimizer_choice = hp.Choice('optimizer', ['adam', 'rmsprop', 'sgd'])
    loss_choice = hp.Choice('loss', ['binary_crossentropy', 'hinge'])

    model.compile(
        optimizer=optimizer_choice,
        loss=loss_choice,
        metrics=['accuracy']
    )
    return model

# Initialize the RandomSearch tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='kt_dir',
    project_name='churn_tuning'
)

print("Searching for best hyperparameters...")
tuner.search(X_train, y_train, epochs=10, validation_split=0.2, verbose=1)

# Get the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBest Hyperparameters Found:\nLayers: {best_hps.get('num_layers')}\nOptimizer: {best_hps.get('optimizer')}\nLoss: {best_hps.get('loss')}")

best_tuned_model = tuner.hypermodel.build(best_hps)
history_tuned = best_tuned_model.fit(X_train, y_train, epochs=50, validation_split=0.2, verbose=0)

# Evaluate optimized model
tuned_loss, tuned_acc = best_tuned_model.evaluate(X_test, y_test)
print(f"\nOptimized ANN Accuracy: {tuned_acc:.4f}")

Model Comparison and Final Selection

In [ ]:
import pandas as pd

# Prepare the final comparison data
final_data = {
    'Model': ['Tuned LightGBM', 'Tuned XGBoost', 'Optimized ANN'],
    'Test Accuracy': [tuned_lgbm_acc, tuned_xgb_acc, tuned_acc]
}

final_summary_df = pd.DataFrame(final_data).sort_values(by='Test Accuracy', ascending=False)

print("--- BEST HYPERPARAMETERS ---")
print(f"LightGBM: {lgbm_grid.best_params_}")
print(f"XGBoost: {xgb_grid.best_params_}")
print(f"ANN: Layers={best_hps.get('num_layers')}, Optimizer={best_hps.get('optimizer')}, Loss={best_hps.get('loss')}")

print("\n--- FINAL MODEL COMPARISON ---")
display(final_summary_df)

best_model_name = final_summary_df.iloc[0]['Model']
print(f"\nBased on the Test Accuracy, the best model is: {best_model_name}")

### Executive Summary: Bank Customer Churn Prediction

**1. Objective:**
The goal of this project was to build a predictive system to identify customers likely to churn (exit) from a bank. This involves a complete end-to-end ML and Deep Learning pipeline.

**2. Data Preprocessing & EDA:**
*   **Insights:** EDA revealed that older customers and those with a higher balance tended to churn more frequently.
*   **Cleaning:** Dropped irrelevant identifiers (`RowNumber`, `CustomerId`, `Surname`).
*   **Engineering:** Applied Label Encoding for Gender and One-Hot Encoding for Geography.
*   **Leakage Prevention:** Features were scaled using `StandardScaler` fitted *only* on the training data.
*   **Outlier Management:** Capped extreme values in `Age` and `CreditScore` to stabilize model training.

**3. Handling Class Imbalance:**
*   Since the dataset was imbalanced (~20% churn), **SMOTE** (Synthetic Minority Over-sampling Technique) was used during the training phase to balance classes, which significantly improved the **Recall** and **ROC-AUC** scores across traditional ML models.

**4. Machine Learning Benchmarking:**
*   Evaluated 8 models (Logistic Regression, Decision Tree, RF, GBM, XGBoost, LightGBM, SVM, KNN).
*   **LightGBM** and **XGBoost** were identified as the top-performing traditional models and were subsequently tuned using `GridSearchCV`.

**5. Deep Learning (ANN) & Optimization:**
*   **Architecture:** Built a multi-layer Sequential ANN with Dropout (0.2) to prevent overfitting.
*   **Callbacks:** Integrated `EarlyStopping` (to halt training when validation loss stops improving) and `ModelCheckpoint` (to save the best weights).
*   **KerasTuner:** Conducted a `RandomSearch` to find optimal hyperparameters for the number of layers, units, activation functions, and optimizers.

**6. Final Performance Comparison:**

| Model | Test Accuracy | Status |
| :--- | :--- | :--- |
| **Optimized ANN** | **86.05%** | **Winner** |
| Tuned XGBoost | 85.95% | Top ML |
| Tuned LightGBM | 85.90% | Runner-up |

**Conclusion:**
The **Optimized ANN** provided the highest accuracy, making it the recommended model for deployment to predict customer churn with high reliability.

c)	Use the following Accident detection Dataset link and create an end-to-end project to predict Accident has happened or not.

https://www.kaggle.com/datasets/ckay16/accident-detection-from-cctv-footage/data

i.	Import the dataset and perform the EDA and do the visualizations.

ii.	Create model from scratch using your own number of filters, loss functions, no. of epochs and check the accuracy.

iii.	Now use the pretrained model to check if accuracy gets improved.

iv.	After that, go for data augmentation and further check if accuracy has improved.

v.	Compare the training and testing accuracy with all these 3 approaches and suggest the best model.


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d ckay16/accident-detection-from-cctv-footage

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/accident-detection-from-cctv-footage.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()


In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# generators
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/data/train',
    labels='inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(256,256)
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/data/test',
    labels='inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(256,256)
)


In [ ]:
# Normalize
def process(image,label):
    image = tf.cast(image/255. ,tf.float32)
    return image,label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

In [ ]:
# Create the CNN model
model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

# model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
# model.add(BatchNormalization())
# model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1,activation='sigmoid'))


In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)

In [ ]:
history = model.fit(train_ds,epochs=25,validation_data=validation_ds,callbacks= [early_stopping , model_checkpoint] )

# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(validation_ds)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()

## VGG16

In [ ]:
from keras.applications.vgg16 import VGG16

# Load the VGG16 model with pre-trained ImageNet weights, excluding the top dense layers
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(256, 256, 3)
)

# Freeze the convolutional base
conv_base.trainable = False

# Build the model
model_vgg = Sequential([
    conv_base,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_vgg.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint('best_model_vgg.h5', monitor='val_loss', save_best_only=True)

# Train the model
history_vgg = model_vgg.fit(train_ds, epochs=25, validation_data=validation_ds, callbacks= [early_stopping,model_checkpoint] )

# Evaluate the model on the test data
test_loss, test_accuracy = model_vgg.evaluate(validation_ds)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

# Visualize accuracy for comparison
plt.plot(history_vgg.history['accuracy'], color='red', label='vgg_train')
plt.plot(history_vgg.history['val_accuracy'], color='blue', label='vgg_validation')
plt.legend()
plt.title('VGG16 Pretrained Model Accuracy')
plt.show()

## Data Augmentation on VGG16

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# Define data augmentation layers
data_augmentation = Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])


# Build a new model incorporating data augmentation
model_aug = Sequential([
    data_augmentation,
    conv_base, # Using the same VGG16 base from previous step
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])


model_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint('best_model_aug.h5', monitor='val_loss', save_best_only=True)

# Train the model with augmented data
history_aug = model_aug.fit(train_ds, epochs=25, validation_data=validation_ds, callbacks= [early_stopping,model_checkpoint])

# Evaluate the model on the test data
test_loss, test_accuracy = model_aug.evaluate(validation_ds)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

# Visualize accuracy for comparison
plt.plot(history_aug.history['accuracy'], color='red', label='aug_train')
plt.plot(history_aug.history['val_accuracy'], color='blue', label='aug_validation')
plt.legend()
plt.title('VGG16 with Data Augmentation Accuracy')
plt.show()

## ResNet50 Pre-trained Model

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50

# Load the ResNet50 model with pre-trained ImageNet weights
resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(256, 256, 3)
)

# Freeze the convolutional base
resnet_base.trainable = False

# Build the model
model_resnet = Sequential([
    resnet_base,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_resnet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Set up callbacks
early_stopping_resnet = EarlyStopping(monitor='val_loss', patience=3)
checkpoint_resnet = ModelCheckpoint('best_model_resnet.h5', monitor='val_loss', save_best_only=True)

# Train the model
history_resnet = model_resnet.fit(train_ds, epochs=25, validation_data=validation_ds, callbacks=[early_stopping_resnet, checkpoint_resnet])

# Evaluate and Visualize
test_loss_res, test_acc_res = model_resnet.evaluate(validation_ds)
print(f"ResNet50 Test Accuracy: {test_acc_res}")

plt.plot(history_resnet.history['accuracy'], color='purple', label='resnet_train')
plt.plot(history_resnet.history['val_accuracy'], color='orange', label='resnet_validation')
plt.legend()
plt.title('ResNet50 Pretrained Model Accuracy')
plt.show()

## Data Augmentation on CNN model

In [ ]:
from tensorflow.keras import layers, Sequential

# 1. Define Data Augmentation specifically for the custom CNN
cnn_augmentation = Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

# 2. Build a new model using the existing 'model' (Custom CNN) and adding augmentation
# We use the 'model' variable which contains your custom architecture
model_cnn_aug = Sequential([
    cnn_augmentation,
    model # This is your custom CNN
])

model_cnn_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint('best_model_aug.h5', monitor='val_loss', save_best_only=True)

# 3. Train the augmented custom CNN
# We'll call this history_cnn_aug to distinguish it
history_cnn_aug = model_cnn_aug.fit(train_ds, epochs=25, validation_data=validation_ds, callbacks = [early_stopping, model_checkpoint])

# Evaluate the model on the test data
test_loss, test_accuracy = model_cnn_aug.evaluate(validation_ds)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")


# 4. Visualize the results for the augmented Custom CNN
plt.plot(history_cnn_aug.history['accuracy'], color='green', label='cnn_aug_train')
plt.plot(history_cnn_aug.history['val_accuracy'], color='orange', label='cnn_aug_validation')
plt.legend()
plt.title('Custom CNN with Data Augmentation Accuracy')
plt.show()

## Overall Accuracy

In [ ]:
import pandas as pd

# Final Summary including ResNet50
final_comparison = {
    'Model Approach': [
        'Custom CNN (Base)',
        'Pre-trained VGG16',
        'VGG16 + Augmentation',
        'Custom CNN + Augmentation',
        'Pre-trained ResNet50'
    ],
    'Train Accuracy': [
        history.history['accuracy'][-1],
        history_vgg.history['accuracy'][-1],
        history_aug.history['accuracy'][-1],
        history_cnn_aug.history['accuracy'][-1],
        history_resnet.history['accuracy'][-1]
    ],
    'Validation Accuracy': [
        history.history['val_accuracy'][-1],
        history_vgg.history['val_accuracy'][-1],
        history_aug.history['val_accuracy'][-1],
        history_cnn_aug.history['val_accuracy'][-1],
        history_resnet.history['val_accuracy'][-1]
    ],
    'Test Accuracy': [
        model.evaluate(validation_ds, verbose=0)[1],
        model_vgg.evaluate(validation_ds, verbose=0)[1],
        model_aug.evaluate(validation_ds, verbose=0)[1],
        model_cnn_aug.evaluate(validation_ds, verbose=0)[1],
        model_resnet.evaluate(validation_ds, verbose=0)[1]
    ]
}

df_final = pd.DataFrame(final_comparison)
display(df_final)

# Final Recommendation
best_approach = df_final.iloc[df_final['Validation Accuracy'].idxmax()]['Model Approach']
print(f"\nThe best overall approach for accident detection is: {best_approach}")

### Final Project Summary and Model Comparison

After evaluating multiple architectures and techniques, we can conclude the following for the CCTV accident detection task:

| Model Approach | Train Accuracy | Validation Accuracy | Test Accuracy |
| :--- | :--- | :--- | :--- |
| **Pre-trained VGG16** | **95.4%** | **95.0%** | **95.0%** |
| VGG16 + Augmentation | 82.2% | 83.0% | 83.0% |
| Custom CNN (Base) | 65.9% | 53.0% | 59.0% |
| Custom CNN + Augmentation | 56.1% | 59.0% | 59.0% |
| Pre-trained ResNet50 | 53.4% | 53.0% | 53.0% |

### Key Insights:
1. **Winner: Pre-trained VGG16** - This model provided the best performance by a significant margin. Transfer learning allowed the model to leverage complex features learned from ImageNet, which were highly relevant to detecting vehicle patterns in CCTV footage.
2. **ResNet50 Performance** - Interestingly, the more complex ResNet50 performed similarly to a random classifier (53%) on this specific binary task, suggesting that its deeper architecture might require more data or fine-tuning to surpass the simpler VGG16 on this small dataset.
3. **Data Augmentation** - In this instance, adding high-variance augmentation (rotations, zooms) actually lowered the accuracy of the VGG16 model. This indicates the original dataset's distribution was likely already well-captured, and the added noise hindered convergence.

### Recommendation:
For real-world deployment on CCTV streams, the **Pre-trained VGG16** model should be used as the backbone due to its high accuracy and robustness.

d)	Explain the role of padding and stride in convolutional layers? How will you decide when to use the padding.

#### 1. Padding
**Padding** involves adding extra pixels (usually zeros) around the boundary of the input image before applying the convolution operation.
*   **Role**: It prevents the spatial dimensions of the feature map from shrinking too quickly as data passes through multiple layers. Without padding, the pixels at the edges are only covered once by the filter, while center pixels are covered multiple times, leading to information loss at the borders.
*   **When to use**:
    *   **Same Padding**: Use this when you want the output feature map to have the same height and width as the input image. It is standard practice in deep networks (like ResNet or VGG) to keep the spatial volume consistent.
    *   **Valid Padding (No Padding)**: Use this when you are okay with the spatial dimensions shrinking or when the boundary information is not critical for the task.

#### 2. Stride
**Stride** is the number of pixels by which the filter shifts over the input image.
*   **Role**: It controls how the filter convolves around the input. A stride of 1 moves the filter one pixel at a time, resulting in a dense feature map. A stride of 2 or more skips pixels, which effectively downsamples the input and reduces the computational load and memory footprint.
*   **Decision**: Large strides are used to reduce spatial resolution instead of (or in addition to) pooling layers, helping the model learn more abstract features by increasing the receptive field.

e)	How does data augmentation help in improving CNN performance?

Data augmentation is a technique used to artificially increase the size of a training dataset by creating modified versions of images in the dataset. It helps improve CNN performance in several key ways:

1.  **Reduces Overfitting**: By presenting the model with different variations of the same image (e.g., flipped, rotated, zoomed), the model is prevented from 'memorizing' specific training examples. This forces the network to learn more robust, global features rather than noise.

2.  **Improves Generalization**: In real-world scenarios, objects can appear in various orientations, lighting conditions, or scales. Augmentation simulates these variations, ensuring the model performs well on unseen data that might differ slightly from the original training set.

3.  **Invariance Learning**: It helps the CNN learn important spatial invariances. For example, **Horizontal Flipping** helps the model learn that a car is still a car whether it's facing left or right (**Translation/Rotation Invariance**).

4.  **Acts as Regularization**: Much like Dropout or L2 regularization, data augmentation acts as a regularizer, making the training objective more difficult and resulting in a more stable and reliable model backbone.

5.  **Compensates for Small Datasets**: In tasks like accident detection where high-quality labeled data might be scarce, augmentation allows us to extract the maximum possible information from the available samples.